In [1]:
import requests

In [2]:
response = requests.get('http://preprocessing-plugin:8003/health')
print(response.json())
print(response.text)

{'status': 'ok'}
{"status":"ok"}


In [3]:
response = requests.get('http://preprocessing-plugin:8003/metadata')
print(response.json())
print(response.text)

{'name': 'Image Classification', 'version': '1.0', 'author': "Based on Karla Sam-Millan's work", 'entry_point': 'plugin.ImageClassificationPlugin', 'module': 'plugins.imageclassification.plugin', 'class': 'ImageClassificationPlugin', 'task': 'classification', 'docker_port': 8003}
{"name":"Image Classification","version":"1.0","author":"Based on Karla Sam-Millan's work","entry_point":"plugin.ImageClassificationPlugin","module":"plugins.imageclassification.plugin","class":"ImageClassificationPlugin","task":"classification","docker_port":8003}


In [4]:
import requests

plugins = [
    "http://preprocessing-plugin:8003"
]

catalog = {}

for plugin in plugins:
    try:
        metadata = requests.get(
            f"{plugin}/metadata"
        ).json()
        catalog[metadata["name"]] = {
            "url": plugin,
            "metadata": metadata,
            "status": "online"
        }
    except Exception as e:
        print(f"ERROR: {plugin}")
        print(e)

In [5]:
catalog

{'Image Classification': {'url': 'http://preprocessing-plugin:8003',
  'metadata': {'name': 'Image Classification',
   'version': '1.0',
   'author': "Based on Karla Sam-Millan's work",
   'entry_point': 'plugin.ImageClassificationPlugin',
   'module': 'plugins.imageclassification.plugin',
   'class': 'ImageClassificationPlugin',
   'task': 'classification',
   'docker_port': 8003},
  'status': 'online'}}

In [6]:
for name, plugin in catalog.items():
    try:
        health = requests.get(
            f"{plugin['url']}/health",
            timeout=2
        ).json()
        plugin["status"] = health["status"]
        print(f"{plugin['url']}, status: {plugin["status"]}")
    except Exception:
        plugin["status"] = "offline"

http://preprocessing-plugin:8003, status: ok


In [7]:
plugin['status']

'ok'

In [8]:
from plugin_manager import PluginManager
plugins = [
    "http://preprocessing-yolo-api:8001",
    "http://preprocessing-maseg-api:8002",
    "http://preprocessing-plugin:8003"
]

manager = PluginManager(plugins)

manager.discover()
manager.get_catalog()

Error descubriendo http://preprocessing-yolo-api:8001: HTTPConnectionPool(host='preprocessing-yolo-api', port=8001): Max retries exceeded with url: /metadata (Caused by NameResolutionError("HTTPConnection(host='preprocessing-yolo-api', port=8001): Failed to resolve 'preprocessing-yolo-api' ([Errno -3] Temporary failure in name resolution)"))
Error descubriendo http://preprocessing-maseg-api:8002: 'name'


{'Image Classification': {'url': 'http://preprocessing-plugin:8003',
  'metadata': {'name': 'Image Classification',
   'version': '1.0',
   'author': "Based on Karla Sam-Millan's work",
   'entry_point': 'plugin.ImageClassificationPlugin',
   'module': 'plugins.imageclassification.plugin',
   'class': 'ImageClassificationPlugin',
   'task': 'classification',
   'docker_port': 8003},
  'status': 'online'}}

In [9]:
from inference_service import InferenceService

service = InferenceService(manager)

In [10]:
import numpy as np

image = np.random.randint(
    0,
    255,
    size=(512, 512),
    dtype=np.uint8
)

response = service.predict(
    "Image Classification",
    {
        "image": image.tolist()
    }
)

{'url': 'http://preprocessing-plugin:8003', 'metadata': {'name': 'Image Classification', 'version': '1.0', 'author': "Based on Karla Sam-Millan's work", 'entry_point': 'plugin.ImageClassificationPlugin', 'module': 'plugins.imageclassification.plugin', 'class': 'ImageClassificationPlugin', 'task': 'classification', 'docker_port': 8003}, 'status': 'online'}


In [19]:
print(response)

{'shape': [512, 512]}
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None


In [ ]:
import cv2
import base64

# image = cv2.imread("imagen.png")
_, buffer = cv2.imencode(".png", image)

payload = {
    "image": base64.b64encode(
        buffer.tobytes()
    ).decode()
}

In [16]:
import threading
import time

def monitor_plugins(manager):
    while True:
        print(manager.refresh_status())
        time.sleep(30)

In [17]:
thread = threading.Thread(
    target=monitor_plugins,
    args=(manager,),
    daemon=True
)

thread.start()

None


In [18]:
manager

None
None
None
None
None
